# Obravnava manjkajočih vrednosti — B: vaja (spletna trgovina)
**Akademija UI za poslovne aplikacije — Modul 1, popoldanski sklop**

Predavatelj je v zvezku `A_manjkajoce_vrednosti` postopek pokazal na telekomu.
Zdaj obravnavate en numerični in en kategorični stolpec spletne trgovine in **zapišete, zakaj** ste izbrali tak pristop.

### Ponovitev
- **Kako manjka:** `NaN` (Pandas ga prepozna sam) ali **posebne oznake** (`"U"`, `"UNKW"`, `-1`, `999`, `1900-01-01`),
  ki jih moramo Pandasu povedati: `read_csv(na_values=...)`.
- **Manjkajoča vrednost ni ničla:** 0 nakupov je podatek, `NaN` pomeni, da vrednosti ne poznamo.
- Najprej **delež** (`df.isna().mean()`), nato **vzorec**: manjka naključno ali pri določenih strankah / v določenem obdobju?
- **Možnosti obravnave:**

| Metoda | Kdaj | Tveganje |
|---|---|---|
| Odstranitev vrstic | Manjka malo in naključno | Izgubimo podatke, pristranskost |
| Odstranitev stolpca | Manjka skoraj vse | Izgubimo informacijo |
| Mediana / povprečje | Numerični stolpec | Umetna konica, manjša razpršenost |
| Najpogostejša kategorija | Kategorični, manjka malo | Napihne največjo skupino |
| Oznaka »neznano« | Kategorični stolpec | Nova kategorija |
| Indikator manjkanja | Manjkanje nosi informacijo | En stolpec več |

- **Varovalka:** mediano in druge parametre izračunamo samo na **učnih** podatkih.
- **Univerzalno pravilne metode za nadomeščanje ni** — pomemben je razlog.

### Nastavitev okolja
Zaženite spodnjo celico (izberite jo in pritisnite **Shift + Enter**). Ničesar ne spreminjajte.
Celica deluje lokalno in v Google Colabu. Koda je skrita — odprete jo s klikom na celico.

In [ ]:
#@title Nastavitev okolja (kliknite za prikaz kode)
import subprocess, sys
from pathlib import Path

REPO_URL = "https://github.com/valira-ai/ui-za-poslovne-aplikacije.git"

if "google.colab" in sys.modules:           # Google Colab: prenesemo gradivo z GitHuba
    KOREN = Path("/content/ui-za-poslovne-aplikacije")
    if not KOREN.exists():
        subprocess.run(["git", "clone", "-q", REPO_URL, str(KOREN)], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "plotnine>=0.15", "kagglehub"], check=True)
else:                                       # lokalno: poiščemo mapo, v kateri je utils/orodja.py
    KOREN = Path.cwd()
    while not (KOREN / "utils/orodja.py").exists() and KOREN != KOREN.parent:
        KOREN = KOREN.parent
sys.path.insert(0, str(KOREN))              # da Python najde utils/orodja.py
print("Gradivo je v:", KOREN)

### Podatki
Funkcija `poskrbi_za_podatke()` iz `utils/orodja.py` ob prvem zagonu prenese podatke s Kaggla (~1 minuta)
in jih shrani v mapo `data/`. Kako to deluje, smo pokazali v zvezku `A_prvi_stik_s_podatki`.

In [ ]:
from utils.orodja import poskrbi_za_podatke
DATA = poskrbi_za_podatke()

### Knjižnice in videz grafov
`pripravi_narocila` združi tabele trgovine v eno (koda je v zvezku `A_vizualizacija`),
`primerjaj_pred_po` nariše porazdelitev pred in po obravnavi (koda je v zvezku `A_manjkajoce_vrednosti`).

In [ ]:
import numpy as np
import pandas as pd
from plotnine import *

from utils.orodja import nastavi_slog, MODRA, ORANZNA, ZELENA, SIVA, BARVE_ODHOD
from utils.orodja import pripravi_narocila, primerjaj_pred_po
nastavi_slog()      # enoten videz grafov (koda je v zvezku A_vizualizacija)

---
# Del B — Vaja: spletna trgovina
Ena vrstica = eno naročilo. Na vrhu celice zamenjate ime stolpca in celico ponovno zaženete (**Shift + Enter**).

In [ ]:
narocila = pripravi_narocila(DATA)
narocila.isna().sum().sort_values(ascending=False).head(10)

## B1. Pri katerih naročilih manjka?
Delež manjkajočih po **skupinah** (`groupby`) — na primer po statusu naročila.

Poskusite: `STOLPEC = "dni_dostave"`, `"vrednost"`, `"ocena"`, `"kategorija"` in `SKUPINA = "status"` ali `"zvezna_drzava"`.

**Vprašanje:** Manjkajo vrednosti naključno ali z razlogom?

In [ ]:
STOLPEC = "dni_dostave"       # ← zamenjajte
SKUPINA = "status"            # ← zamenjajte

po_skupinah = (narocila.groupby(SKUPINA)[STOLPEC]
               .agg(manjka=lambda s: s.isna().sum(), vseh="size")
               .reset_index())
po_skupinah["odstotek"] = 100 * po_skupinah["manjka"] / po_skupinah["vseh"]
po_skupinah = po_skupinah.sort_values("odstotek")
print(po_skupinah.round(1).to_string(index=False))

(ggplot(po_skupinah, aes(x=SKUPINA, y="odstotek"))
 + geom_col(fill=MODRA)
 + scale_x_discrete(limits=po_skupinah[SKUPINA].tolist())
 + coord_flip()
 + labs(title=f"% manjkajočih v '{STOLPEC}' glede na '{SKUPINA}'", x="", y="% manjkajočih"))

Še po **času**: `resample("ME")` združi vrstice po mesecih (potrebuje datum v indeksu, zato `set_index`).

In [ ]:
STOLPEC = "dni_dostave"       # ← zamenjajte

po_mesecih = (narocila.set_index("datum_nakupa").sort_index()[STOLPEC]
              .isna().resample("ME").mean().mul(100)
              .reset_index(name="odstotek"))
(ggplot(po_mesecih, aes(x="datum_nakupa", y="odstotek"))
 + geom_line(color=MODRA)
 + geom_point(color=MODRA)
 + labs(title=f"% manjkajočih v '{STOLPEC}' po mesecih", x="", y="% manjkajočih"))

<details>
<summary><b>Rešitev (klikni)</b></summary>

- `dni_dostave` manjka skoraj samo pri naročilih, ki **niso dostavljena** (`shipped`, `canceled`, `unavailable` …).
  Po mesecih: na koncu podatkov (september, oktober 2018) manjka 100 % — naročila so bila še na poti, ko so podatke izvozili.
  Manjka **z razlogom**: dostave ni bilo (še).
- `vrednost` (in `teza_g`, `postnina`) manjka pri naročilih brez postavk — večinoma `unavailable` in `canceled`.
- `kategorija` manjka pri ~2 % naročil: pri naročilih brez postavk in pri izdelkih, ki nimajo vpisane kategorije.

</details>

## B2. Obravnavajte en numerični stolpec
Spodnja celica izračuna **vse tri možnosti** iz dela A. Zamenjajte `STOLPEC`, poglejte izpis in graf,
nato v B4 izberite, katera je po vašem najboljša.

In [ ]:
STOLPEC = "dni_dostave"       # ← zamenjajte: "teza_g", "vrednost", "ocena"

s = narocila[STOLPEC].astype(float)
odstranjeno = s.dropna()                      # možnost 1: odstranimo vrstice
z_mediano = s.fillna(s.median())              # možnost 2: mediana
indikator = s.isna().astype(int)              # možnost 3: mediana + indikator (nov stolpec)

print(f"Odstranitev: ostane {len(odstranjeno)} od {len(s)} vrstic")
print(f"Mediana:     vse vrstice; manjkajoče dobijo vrednost {s.median()}")
print(f"Indikator:   {indikator.sum()} vrstic dobi oznako 1 = 'manjkalo'")
print(f"Std. odklon: {s.std():.2f} → z mediano {z_mediano.std():.2f}")

primerjaj_pred_po(s, z_mediano, f"{STOLPEC}: nadomeščanje z mediano")

## B3. Obravnavajte en kategorični stolpec

In [ ]:
STOLPEC = "kategorija"        # ← zamenjajte: "nacin_placila"

najpogostejsa = narocila[STOLPEC].mode()[0]
print("Manjkajočih:", narocila[STOLPEC].isna().sum())
print("Najpogostejša vrednost:", najpogostejsa)

pd.DataFrame({
    "fillna(najpogostejša)": narocila[STOLPEC].fillna(najpogostejsa).value_counts().head(5),
    "fillna('neznano')":     narocila[STOLPEC].fillna("neznano").value_counts().head(5),
})

## B4. Zapišite svojo odločitev
Dvakrat kliknite to celico in dopolnite. Ni enega pravilnega odgovora — pomemben je **razlog**.

**Numerični stolpec:** ...
**Metoda:** ...
**Zakaj:** ...

**Kategorični stolpec:** ...
**Metoda:** ...
**Zakaj:** ...

<details>
<summary><b>Rešitev (klikni)</b></summary>

Primer razmisleka (ne edini pravilni):

**`dni_dostave`** — za analizo časa dostave vrstice brez dostave **odstranimo** (oz. obdržimo samo `delivered`):
naročilo, ki ni bilo dostavljeno, nima časa dostave. Mediana (10 dni) bi si izmislila dostave, ki se niso zgodile.
Ko bomo čas dostave napovedovali, so to vrstice brez ciljne vrednosti — te pri učenju modela izpustimo.
Če gradimo model, npr. za napoved slabe ocene, je boljši **indikator** — "ni dostavljeno" je močan signal.

**`teza_g`** — manjka pri naročilih brez postavk. Ta naročila so prazna (preklicana), zato jih pri analizi teže
**odstranimo**. Mediana bi jim dodala izmišljen paket.

**`kategorija`** — manjka pri ~2 %. Uporabimo **`neznano`**: ohranimo naročila in ne napihnemo najpogostejše
kategorije (`bed_bath_table`). Pri tako majhnem deležu bi bila sprejemljiva tudi najpogostejša vrednost,
a `neznano` je poštenejše.

**`nacin_placila`** — manjka pri enem naročilu. Tu je vseeno: najpogostejša vrednost ali odstranitev.

</details>

---
### Za hitre
1. Primerjajte mediano in povprečje za `vrednost`: `narocila["vrednost"].median()`, `narocila["vrednost"].mean()`. Zakaj sta tako različni?
2. Kaj se zgodi z `ocena`, če manjkajoče nadomestite s povprečjem? Lahko ima naročilo oceno 4,1?